# 📊 Unidad 4: Proyectos Integradores
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Contenido Teórico

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta unidad teórica, serás capaz de:

1. ✅ Comprender la metodología de proyectos integradores end-to-end
2. ✅ Diseñar pipelines de datos completos (ETL + transformación + análisis)
3. ✅ Aplicar arquitecturas Delta Lake (Bronze-Silver-Gold)
4. ✅ Integrar análisis descriptivo, predictivo y prescriptivo
5. ✅ Desarrollar dashboards y presentaciones ejecutivas
6. ✅ Implementar mejores prácticas de ingeniería de datos y ML
7. ✅ Documentar y comunicar resultados de forma profesional

---

### 📚 Contenido

1. Introducción a Proyectos Integradores
2. Arquitectura de Datos: Medallion (Bronze-Silver-Gold)
3. Pipeline ETL End-to-End
4. Feature Engineering para Machine Learning
5. Análisis Predictivo y Prescriptivo
6. Dashboards Ejecutivos
7. Documentación y Presentación de Proyectos
8. Mejores Prácticas de Producción

---

### ⏱️ Duración Estimada: 3 horas

## 1️⃣ Introducción a Proyectos Integradores

### ¿Qué es un Proyecto Integrador?

Un **proyecto integrador** es un ejercicio práctico que:
* **Combina múltiples disciplinas**: Ingeniería de datos, análisis, ML, visualización
* **Simula escenarios reales**: Problemas de negocio con datos reales o realistas
* **Requiere toma de decisiones**: No hay un único camino correcto
* **End-to-end**: Desde datos crudos hasta insights accionables

### Características de un Buen Proyecto Integrador

#### 🎯 **1. Problema de Negocio Claro**
* Pregunta bien definida
* Métricas de éxito identificadas
* Stakeholders y audiencia conocidos
* Impacto de negocio cuantificable

#### 📁 **2. Datos Realistas**
* Volumen representativo (no trivial)
* Calidad imperfecta (valores faltantes, outliers)
* Múltiples fuentes que requieren integración
* Dimensión temporal (series de tiempo)

#### 🔧 **3. Complejidad Técnica**
* ETL con transformaciones no triviales
* Feature engineering creativo
* Modelado (descriptivo o predictivo)
* Optimización de performance

#### 📊 **4. Entregables Profesionales**
* Código limpio y documentado
* Notebooks narrativos (storytelling)
* Dashboards interactivos
* Presentación ejecutiva
* Recomendaciones accionables

### Ejemplos de Proyectos Integradores

**E-commerce:**
* Análisis de churn y estrategias de retención
* Optimización de inventario y forecasting
* Segmentación de clientes y personalización

**Finanzas:**
* Detección de fraude transaccional
* Scoring crediticio y riesgo
* Predicción de morosidad

**Retail:**
* Análisis de ventas y optimización de precios
* Market basket analysis
* Forecast de demanda por producto/tienda

**Manufactura:**
* Mantenimiento predictivo de equipos
* Control de calidad automatizado
* Optimización de cadena de suministro

## 2️⃣ Arquitectura de Datos: Medallion (Bronze-Silver-Gold)

### ¿Qué es la Arquitectura Medallion?

La **arquitectura Medallion** organiza datos en tres capas progresivamente refinadas:

```
🟤 BRONZE (Raw)         🥈 SILVER (Cleaned)      🥇 GOLD (Business)
│                        │                         │
│ - Datos crudos         │ - Datos limpios         │ - Agregaciones
│ - Sin transformar      │ - Validados             │ - Métricas de negocio
│ - Histórico completo   │ - Deduplicated          │ - Features de ML
│ - Schema on read       │ - Schema enforced       │ - Optimizado para queries
│                        │                         │
└───────> ETL ──────────>│───> Transformación ────>│   Limpieza  Enriquecimiento
```

### Capa Bronze 🟤 (Raw / Landing)

**Propósito**: Almacenar datos crudos tal como llegan de las fuentes

**Características:**
* Copia exacta de la fuente (fidelidad completa)
* Formato original (CSV, JSON, Parquet, logs, APIs)
* Sin transformaciones (sólo carga)
* Histórico completo (nunca se elimina)
* Inmutable (append-only)

**Estructura típica:**
```
bronze/
  ├── ventas_raw/         (particionado por fecha de ingesta)
  ├── clientes_raw/
  ├── productos_raw/
  └── logs_raw/
```

**Ejemplo en Databricks:**
```python
# Carga a Bronze sin transformaciones
df_raw = (spark.read
  .format("csv")
  .option("header", "true")
  .load("/mnt/source/ventas.csv"))

df_raw.write.format("delta").mode("append").save("/bronze/ventas_raw")
```

### Capa Silver 🥈 (Cleaned / Curated)

**Propósito**: Datos limpios, validados y listos para análisis

**Transformaciones aplicadas:**
* ✅ Limpieza de nulos y valores inválidos
* ✅ Deduplicación
* ✅ Schema enforcement (tipos correctos)
* ✅ Estandarización de formatos (fechas, textos)
* ✅ Joins entre fuentes relacionadas
* ✅ Filtrado de registros corruptos

**Estructura:**
```
silver/
  ├── ventas_clean/       (particionado por fecha de venta)
  ├── clientes_clean/
  ├── productos_clean/
  └── transacciones/      (joins aplicados)
```

**Ejemplo:**
```python
# Bronze → Silver: Limpieza y validación
df_silver = (df_bronze
  .dropDuplicates(["id_transaccion"])
  .filter(col("monto") > 0)
  .withColumn("fecha", to_date(col("fecha_str"), "yyyy-MM-dd"))
  .na.fill({"descuento": 0}))

df_silver.write.format("delta").mode("overwrite").save("/silver/ventas_clean")
```

### Capa Gold 🥇 (Business / Aggregated)

**Propósito**: Datos optimizados para casos de uso específicos

**Transformaciones:**
* 📊 Agregaciones de negocio (KPIs)
* 🧠 Feature engineering para ML
* 🔗 Joins complejos y denormalización
* 📅 Series temporales y rolling metrics
* 🎯 Vistas materializadas para dashboards

**Estructura por caso de uso:**
```
gold/
  ├── kpis_ventas/           (para dashboards ejecutivos)
  ├── features_ml/            (para modelos de ML)
  ├── rfm_clientes/           (segmentación)
  └── forecast_demanda/       (series temporales)
```

**Ejemplo:**
```python
# Silver → Gold: Agregaciones de negocio
df_gold_kpis = (df_silver
  .groupBy("mes", "categoria")
  .agg(
    sum("monto").alias("ingresos_totales"),
    count("id_transaccion").alias("num_transacciones"),
    countDistinct("id_cliente").alias("clientes_unicos"),
    avg("monto").alias("ticket_promedio")
  ))

df_gold_kpis.write.format("delta").mode("overwrite").save("/gold/kpis_ventas")
```

### Ventajas de la Arquitectura Medallion

✅ **Trazabilidad**: Siempre puedes volver a Bronze y reprocesar  
✅ **Modularidad**: Cada capa tiene un propósito claro  
✅ **Performance**: Gold optimizado para queries rápidos  
✅ **Calidad**: Validaciones progresivas en cada capa  
✅ **Colaboración**: Equipos trabajan en diferentes capas sin conflictos  
✅ **Gobernanza**: Control de acceso por capa (Bronze: restringido, Gold: amplio)

## 3️⃣ Pipeline ETL End-to-End

### Fases del Pipeline

#### **1️⃣ Extract (Extracción)**

**Fuentes comunes:**
* 📁 Archivos (CSV, JSON, Parquet, Excel)
* 📊 APIs REST
* 📦 Bases de datos (SQL, NoSQL)
* 🔄 Streaming (Kafka, Event Hubs)
* ☁️ Cloud storage (S3, ADLS, GCS)

**Mejores prácticas:**
* Validar disponibilidad de fuente antes de extraer
* Manejar errores de conexión con reintentos
* Loggear metadata (timestamp, tamaño, filas)
* Incremental cuando sea posible (no full reload)

#### **2️⃣ Transform (Transformación)**

**Transformaciones típicas:**

**Limpieza:**
```python
# Manejo de nulos
df = df.fillna({"columna1": 0, "columna2": "desconocido"})

# Remoción de duplicados
df = df.dropDuplicates(["id"])

# Filtrado de inválidos
df = df.filter((col("edad") >= 0) & (col("edad") <= 120))
```

**Enriquecimiento:**
```python
# Derivar nuevas columnas
df = df.withColumn("anio", year(col("fecha")))
df = df.withColumn("rango_edad", 
  when(col("edad") < 25, "Joven")
  .when(col("edad") < 50, "Adulto")
  .otherwise("Mayor"))

# Joins
df_enriquecido = df_ventas.join(df_clientes, "id_cliente", "left")
```

**Agregaciones:**
```python
# KPIs por periodo
df_kpis = df.groupBy("mes", "categoria").agg(
  sum("monto").alias("ingresos"),
  count("*").alias("transacciones")
)
```

#### **3️⃣ Load (Carga)**

**Estrategias de carga:**

**Append (Añadir):**
```python
df.write.format("delta").mode("append").save("/ruta/tabla")
```
Uso: Datos nuevos sin duplicados, logs, eventos

**Overwrite (Sobrescribir):**
```python
df.write.format("delta").mode("overwrite").save("/ruta/tabla")
```
Uso: Tablas dimensión, agregaciones recalculadas

**Merge (Upsert):**
```python
# Delta Lake merge para actualizaciones
target_table.alias("target").merge(
  source_df.alias("source"),
  "target.id = source.id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
```
Uso: Dimensiones que cambian lentamente (SCD Type 1)

### Orquestación del Pipeline

**Databricks Jobs:**
```
Job: pipeline_ventas_diario
├── Task 1: extract_to_bronze (paralelo)
│   ├── extract_ventas
│   ├── extract_clientes
│   └── extract_productos
├── Task 2: bronze_to_silver (secuencial, depende Task 1)
│   ├── clean_ventas
│   ├── clean_clientes
│   └── join_transacciones
└── Task 3: silver_to_gold (depende Task 2)
    ├── compute_kpis
    └── build_features_ml
```

**Schedule:**
* Diario a las 2 AM
* Notificaciones por email on failure
* Retry automático (max 3 intentos)

### Monitoreo y Alertas

**Métricas clave a rastrear:**
* ⏱️ Duración de cada task
* 📄 Filas procesadas (input vs output)
* ⚠️ Errores y excepciones
* 💾 Uso de recursos (memoria, CPU)
* ✅ Data quality metrics (nulos, duplicados)

**Alertas:**
```python
# Validaciones post-carga
count_after = spark.read.format("delta").load("/gold/kpis").count()

if count_after == 0:
  raise Exception("⚠️ Tabla Gold vacía - pipeline fallido")

if count_after < count_yesterday * 0.5:
  print("⚠️ Warning: Reducción significativa en datos")
```

## 4️⃣ Feature Engineering para Machine Learning

### ¿Qué es Feature Engineering?

**Feature engineering** es el arte de transformar datos crudos en variables (features) que:
* Capturan patrones relevantes para el problema
* Mejoran el desempeño de modelos ML
* Son interpretables y accionables

> “Los datos mejores modelos provienen de mejores features, no de mejores algoritmos”

### Tipos de Features

#### **1. Features Temporales** 📅

```python
# Extraer componentes de fecha
df = df.withColumn("dia_semana", dayofweek(col("fecha")))
df = df.withColumn("mes", month(col("fecha")))
df = df.withColumn("trimestre", quarter(col("fecha")))
df = df.withColumn("es_fin_semana", col("dia_semana").isin([1, 7]).cast("int"))

# Diferencias temporales
df = df.withColumn("dias_desde_ultima_compra", 
  datediff(col("fecha_actual"), col("fecha_ultima_compra")))

# Agregaciones por ventana temporal (rolling)
window_spec = Window.partitionBy("id_cliente").orderBy("fecha").rowsBetween(-30, 0)
df = df.withColumn("ventas_ultimos_30dias", sum("monto").over(window_spec))
```

#### **2. Features de Agregación** 📊

```python
# Estadísticas por grupo
df_features = df.groupBy("id_cliente").agg(
  count("id_transaccion").alias("num_compras"),
  sum("monto").alias("total_gastado"),
  avg("monto").alias("ticket_promedio"),
  stddev("monto").alias("variabilidad_gasto"),
  min("fecha").alias("primera_compra"),
  max("fecha").alias("ultima_compra")
)

# Frecuencia, Recencia, Monetario (RFM)
df_rfm = df.withColumn(
  "recencia", datediff(current_date(), col("ultima_compra"))
).withColumn(
  "frecuencia", col("num_compras")
).withColumn(
  "monetario", col("total_gastado")
)
```

#### **3. Features Categóricas** 🏷️

**One-Hot Encoding:**
```python
from pyspark.ml.feature import OneHotEncoder, StringIndexer

indexer = StringIndexer(inputCol="categoria", outputCol="categoria_idx")
encoder = OneHotEncoder(inputCol="categoria_idx", outputCol="categoria_vec")
```

**Target Encoding (media del target por categoría):**
```python
# Media de tasa de churn por segmento
target_encoding = df.groupBy("segmento").agg(
  avg("churn").alias("tasa_churn_segmento")
)

df = df.join(target_encoding, "segmento", "left")
```

#### **4. Features de Interacción** 🔗

```python
# Multiplicación de features
df = df.withColumn("precio_x_cantidad", col("precio") * col("cantidad"))

# Ratios
df = df.withColumn("descuento_pct", col("descuento") / col("precio"))
df = df.withColumn("margen_ratio", col("precio_venta") / col("costo"))

# Polinomiales
df = df.withColumn("edad_cuadrado", col("edad") ** 2)
```

#### **5. Features de Texto** 📝

```python
from pyspark.ml.feature import Tokenizer, CountVectorizer, IDF

# Tokenización
tokenizer = Tokenizer(inputCol="descripcion", outputCol="palabras")

# Bag of Words
cv = CountVectorizer(inputCol="palabras", outputCol="bow")

# TF-IDF
idf = IDF(inputCol="bow", outputCol="tfidf")
```

### Escalado y Normalización

```python
from pyspark.ml.feature import StandardScaler, MinMaxScaler

# Estandarización (media 0, desv 1)
scaler = StandardScaler(inputCol="features", outputCol="features_scaled")

# Normalización (rango 0-1)
min_max = MinMaxScaler(inputCol="features", outputCol="features_normalized")
```

### Selección de Features

**Métodos:**

1. **Correlation-based**: Remover features altamente correlacionadas
2. **Feature importance**: De modelos tree-based
3. **Recursive Feature Elimination**: Iterativo
4. **Domain knowledge**: Experiencia del negocio

```python
# Feature importance de Random Forest
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol="features", labelCol="label")
model = rf.fit(train_df)

importances = model.featureImportances
print(f"Feature importances: {importances}")
```

## 5️⃣ Análisis Predictivo y Prescriptivo

### Análisis Predictivo 🔮

**Objetivo**: Predecir eventos futuros basados en datos históricos

#### **Casos de Uso Comunes**

**1. Clasificación Binaria**
* Churn (cliente se va o se queda)
* Fraude (transacción fraudulenta o legítima)
* Default (cliente paga o no paga)

```python
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

# Regresión Logística
lr = LogisticRegression(featuresCol="features", labelCol="churn")
model_lr = lr.fit(train_df)

# Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="churn", numTrees=100)
model_rf = rf.fit(train_df)

# Predicciones
predictions = model_rf.transform(test_df)
```

**2. Clasificación Multiclase**
* Segmentación de clientes (A, B, C, D)
* Categorización de productos
* Nivel de riesgo (bajo, medio, alto)

**3. Regresión**
* Forecasting de ventas
* Predicción de precio
* Tiempo de entrega

```python
from pyspark.ml.regression import LinearRegression, GBTRegressor

# Regresión Lineal
lr = LinearRegression(featuresCol="features", labelCol="ventas")
model = lr.fit(train_df)

print(f"RMSE: {model.summary.rootMeanSquaredError}")
print(f"R2: {model.summary.r2}")
```

#### **Evaluación de Modelos**

**Clasificación:**
```python
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# AUC-ROC
evaluator_auc = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predictions)

# Accuracy, Precision, Recall, F1
evaluator_acc = MulticlassClassificationEvaluator(labelCol="churn", metricName="accuracy")
accuracy = evaluator_acc.evaluate(predictions)
```

**Regresión:**
```python
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(labelCol="ventas", predictionCol="prediction")
rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
mae = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})
```

### Análisis Prescriptivo 💡

**Objetivo**: Recomendar acciones óptimas basadas en predicciones

#### **Enfoque 1: Rules-Based**

```python
# Reglas de negocio basadas en predicciones
df_actions = predictions.withColumn(
  "accion_recomendada",
  when((col("prob_churn") > 0.7) & (col("valor_cliente") > 1000), "Oferta Premium")
  .when((col("prob_churn") > 0.7) & (col("valor_cliente") > 500), "Descuento 20%")
  .when(col("prob_churn") > 0.5, "Email Retención")
  .otherwise("Sin Acción")
)

# Priorizar por impacto esperado
df_priorizado = df_actions.withColumn(
  "valor_en_riesgo", col("prob_churn") * col("valor_cliente")
).orderBy(desc("valor_en_riesgo"))
```

#### **Enfoque 2: Optimización**

**Problema**: Asignar recursos limitados para maximizar retorno

```python
# Ejemplo: Optimizar campañas de marketing
# Restricción: Presupuesto máximo
# Objetivo: Maximizar conversiones esperadas

from scipy.optimize import linprog

# Coeficientes: -prob_conversion (negativo porque linprog minimiza)
c = -df["prob_conversion"].values

# Restricción: costo de contactar cada cliente <= presupuesto
A_ub = [[df["costo_contacto"].values]]
b_ub = [presupuesto_total]

# Resolver
result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=(0, 1), method="highs")

# Clientes a contactar
clientes_seleccionados = result.x > 0.5
```

#### **Enfoque 3: Simulación (What-If Analysis)**

```python
# Simular diferentes escenarios
escenarios = [
  {"descuento": 0.10, "nombre": "Conservador"},
  {"descuento": 0.20, "nombre": "Moderado"},
  {"descuento": 0.30, "nombre": "Agresivo"}
]

resultados = []
for escenario in escenarios:
  # Aplicar descuento simulado
  df_sim = df.withColumn("precio_final", col("precio") * (1 - escenario["descuento"]))
  
  # Re-predecir con nuevo precio
  predictions_sim = model.transform(df_sim)
  
  # Calcular métricas
  ingresos = predictions_sim.selectExpr("sum(precio_final * prob_compra) as ingresos").collect()[0][0]
  resultados.append({"escenario": escenario["nombre"], "ingresos_esperados": ingresos})

# Comparar escenarios
df_resultados = spark.createDataFrame(resultados)
display(df_resultados.orderBy(desc("ingresos_esperados")))
```

### Integración en el Proyecto

**Pipeline completo:**

1. **Entrenar modelo predictivo** en datos históricos (Silver/Gold)
2. **Aplicar predicciones** a datos actuales
3. **Generar recomendaciones** (prescriptivas) basadas en predicciones
4. **Dashboard ejecutivo** mostrando:
   * Top clientes en riesgo de churn
   * Acciones recomendadas priorizadas
   * ROI esperado de cada acción
5. **Monitoreo continuo**: Re-entrenar modelo periódicamente

## 6️⃣ Dashboards Ejecutivos

### Principios de un Dashboard Ejecutivo

**Características clave:**
* 🎯 **Enfocado**: Una historia, no 20 métricas inconexas
* 👁️ **Escaneable**: Insights clave visibles en 5 segundos
* 📊 **Accionable**: Responde "y ahora qué?"
* 📅 **Actualizado**: Timestamp claro, datos frescos
* 🔄 **Interactivo**: Filtros para drill-down

### Estructura Recomendada

```
┌────────────────────────────────────────────────────────────┐
│  TÍTULO DEL DASHBOARD          📅 Última actualización  │
├─────────────────────────────────────────────────────-─┤┤
│  Filtros: [📅 Rango Fecha] [🏪 Región] [📋 Categoría] │
├───────────────────────────────────┤
│                                        │
│  💰 KPIs PRINCIPALES (grande)          │
│  [Ingresos] [Clientes] [Ticket Medio]   │
│  +15%  ↑    +8%  ↑    -2%  ↓          │
│                                        │
├───────────────────────────────────┤
│  📊 GRÁFICOS DE TENDENCIA (mediano)    │
│  [Línea: Ingresos por Mes]             │
│  [Barras: Top Productos]               │
├───────────────────────────────────┤
│  🗒️ DETALLES Y TABLAS (pequeño)        │
│  [Tabla: Transacciones Recientes]      │
└───────────────────────────────────┘
```

### Componentes del Dashboard

#### **1. KPIs con Comparación**

```sql
-- KPI con % cambio vs periodo anterior
SELECT 
  SUM(monto) AS ingresos_actuales,
  LAG(SUM(monto)) OVER (ORDER BY mes) AS ingresos_anterior,
  ROUND((SUM(monto) / LAG(SUM(monto)) OVER (ORDER BY mes) - 1) * 100, 1) AS cambio_pct
FROM gold.ventas
WHERE mes >= DATE_TRUNC('month', CURRENT_DATE - INTERVAL 2 MONTHS)
GROUP BY mes
ORDER BY mes DESC
LIMIT 1
```

**Visualización**: Número grande + indicador de cambio (color verde/rojo, flecha ↑/↓)

#### **2. Gráfico de Tendencia Temporal**

```sql
-- Ingresos mensuales con objetivo
SELECT 
  DATE_TRUNC('month', fecha) AS mes,
  SUM(monto) AS ingresos,
  1000000 AS objetivo  -- Línea de referencia
FROM silver.ventas
WHERE fecha >= CURRENT_DATE - INTERVAL 12 MONTHS
GROUP BY mes
ORDER BY mes
```

**Visualización**: Línea temporal con dos series (real vs objetivo)

#### **3. Comparación Categórica**

```sql
-- Top 10 productos por ingresos
SELECT 
  producto,
  SUM(monto) AS ingresos,
  COUNT(*) AS transacciones
FROM gold.ventas_productos
WHERE fecha >= CURRENT_DATE - INTERVAL 30 DAYS
GROUP BY producto
ORDER BY ingresos DESC
LIMIT 10
```

**Visualización**: Barras horizontales (fácil comparar muchas categorías)

#### **4. Distribución o Composición**

```sql
-- Composición de ingresos por categoría
SELECT 
  categoria,
  SUM(monto) AS ingresos,
  ROUND(SUM(monto) / SUM(SUM(monto)) OVER () * 100, 1) AS porcentaje
FROM gold.ventas_productos
WHERE fecha >= CURRENT_DATE - INTERVAL 30 DAYS
GROUP BY categoria
ORDER BY ingresos DESC
```

**Visualización**: Treemap o barras apiladas (evita pie charts con >5 categorías)

#### **5. Tabla de Detalles**

```sql
-- Últimas transacciones de alto valor
SELECT 
  fecha,
  id_cliente,
  nombre_cliente,
  producto,
  monto
FROM gold.transacciones_detalle
WHERE monto > 1000
ORDER BY fecha DESC
LIMIT 20
```

**Visualización**: Tabla con formato condicional (resaltar valores altos)

### Parámetros Interactivos

**Crear parámetros en Databricks SQL:**

```sql
-- Parámetro de rango de fechas
{{ fecha_inicio }}  -- Widget tipo "date"
{{ fecha_fin }}     -- Widget tipo "date"

-- Parámetro de selección múltiple
{{ regiones }}      -- Widget tipo "multi-select"

-- Uso en query
SELECT *
FROM gold.ventas
WHERE fecha BETWEEN '{{ fecha_inicio }}' AND '{{ fecha_fin }}'
  AND region IN ({{ regiones }})
```

### Checklist de Dashboard Ejecutivo

✅ **¿Los KPIs responden las preguntas clave del negocio?**  
✅ **¿Cada visualización tiene un título descriptivo?**  
✅ **¿Hay comparación temporal (vs periodo anterior)?**  
✅ **¿Los filtros afectan todas las visualizaciones consistentemente?**  
✅ **¿La paleta de colores es consistente y accesible?**  
✅ **¿Hay un call-to-action o recomendación clara?**  
✅ **¿El dashboard carga en <5 segundos?**  
✅ **¿Funciona bien en móvil?**

## 7️⃣ Documentación y Presentación de Proyectos

### Documentación Técnica

#### **README del Proyecto**

```markdown
# Proyecto: Análisis de Churn en E-commerce

## 🎯 Objetivo
Predecir qué clientes tienen mayor riesgo de abandono (churn)
y recomendar acciones de retención priorizadas.

## 📁 Estructura del Proyecto
├── data/
│   ├── bronze/         # Datos crudos
│   ├── silver/         # Datos limpios
│   └── gold/           # Features y agregaciones
├── notebooks/
│   ├── 01_ETL_Bronze_to_Silver.ipynb
│   ├── 02_Feature_Engineering.ipynb
│   ├── 03_Modelado_Predictivo.ipynb
│   └── 04_Dashboard.sql
├── dashboards/
│   └── Dashboard_Churn.dbdash
└── README.md

## 🛠️ Tecnologías
- Databricks Runtime 14.3 LTS
- PySpark 3.5
- MLlib
- Databricks SQL

## 📈 Métricas Clave
- AUC-ROC: 0.87
- Precision: 0.82
- Recall: 0.79
- Clientes identificados en riesgo: 1,247
- Valor en riesgo total: $2.1M

## 🚀 Cómo Ejecutar
1. Clonar el repositorio
2. Importar notebooks a Databricks workspace
3. Ejecutar notebooks en orden (01 → 04)
4. Abrir dashboard para visualizar resultados

## 👥 Equipo
- Data Engineer: [Nombre]
- Data Scientist: [Nombre]
- Business Analyst: [Nombre]
```

#### **Documentación Inline en Notebooks**

```python
# ===================================================
# SECCIÓN: FEATURE ENGINEERING - AGREGACIONES RFM
# ===================================================
# Propósito: Calcular métricas de Recencia, Frecuencia
#            y Monetario por cliente.
# Input:     silver.transacciones
# Output:    gold.features_rfm
# Dependencias: Requiere que silver.transacciones esté actualizado
# ===================================================

from pyspark.sql.functions import count, sum, avg, datediff, max, current_date

# Fecha de referencia para cálculo de recencia
fecha_ref = current_date()

# Calcular RFM
df_rfm = (
    df_transacciones
    .groupBy("id_cliente")
    .agg(
        # Recencia: días desde última compra
        datediff(fecha_ref, max("fecha_compra")).alias("recencia"),
        # Frecuencia: total de compras
        count("id_transaccion").alias("frecuencia"),
        # Monetario: total gastado
        sum("monto").alias("monetario")
    )
)

# Validaciones
assert df_rfm.count() > 0, "⚠️ Error: RFM resultante vacío"
assert df_rfm.filter(col("frecuencia") < 0).count() == 0, "⚠️ Error: Frecuencias negativas"

print(f"✅ Features RFM generados para {df_rfm.count():,} clientes")
```

### Presentación Ejecutiva

#### **Estructura de Presentación (10-15 minutos)**

**Slide 1: Título y Contexto**
* Problema de negocio
* Por qué es importante
* Stakeholders impactados

**Slide 2: Datos y Método**
* Fuentes de datos utilizadas
* Volumen (filas, periodo)
* Arquitectura (Bronze-Silver-Gold)
* Métodos aplicados (brief)

**Slide 3: Insight #1 - Descriptivo**
* Hallazgo principal del análisis exploratorio
* Visualización clara
* Números impactantes

**Slide 4: Insight #2 - Predictivo**
* Predicciones clave
* Precisión del modelo
* Top factores de riesgo/oportunidad

**Slide 5: Recomendaciones (Prescriptivo)**
* 3-5 acciones concretas
* Priorizadas por impacto
* ROI estimado

**Slide 6: Implementación**
* Dashboard disponible (screenshot)
* Automatización (frecuencia de actualización)
* Próximos pasos

**Slide 7: Q&A**
* Contacto
* Links a recursos (notebook, dashboard)

#### **Tips de Presentación**

✅ **Empieza con el "So What?"**: Cuál es el impacto de negocio  
✅ **Una idea por slide**: No sobrecargues  
✅ **Visualizaciones grandes**: Legibles desde el fondo del salón  
✅ **Números redondos**: "2 millones" mejor que "2,143,892"  
✅ **Storytelling**: Inicio (problema) → Desarrollo (análisis) → Final (solución)  
✅ **Practica**: Ensaya el timing (10-15 min estrictos)  
❌ Evita jerga técnica excesiva para audiencias ejecutivas  
❌ No expliques código en la presentación (eso va en notebook)  

### Notebook Narrativo

**Estructura recomendada:**

```
# 1. CONTEXTO DE NEGOCIO
%md
## 🎯 Problema
[Descripción del problema en términos de negocio]

## 📊 Datos Disponibles
[Resumen de fuentes y volumen]

# 2. EXPLORACIÓN DE DATOS
%md
## 🔍 Análisis Exploratorio
[Preguntas clave y hallazgos preliminares]

# Código Python/SQL con comentarios
[...]

# Visualizaciones
display(df.groupBy("categoria").count())

# 3. FEATURE ENGINEERING
%md
## 🛠️ Construcción de Features
[Explicación de features creados y su justificación]

# 4. MODELADO
%md
## 🧠 Modelo Predictivo
[Descripción del enfoque, métricas, interpretación]

# 5. RESULTADOS Y RECOMENDACIONES
%md
## 💡 Insights Clave
1. ...
2. ...

## ✅ Recomendaciones
- ...
```

**Características de un buen notebook narrativo:**
* Celdas markdown frecuentes (no solo código)
* Contexto antes del código (explica el "por qué")
* Visualizaciones con títulos descriptivos
* Conclusiones parciales después de cada sección
* Resumen final con takeaways

## 8️⃣ Mejores Prácticas de Producción

### Código Limpio y Mantenible

#### **Convenciones de Nombres**

```python
# ✅ Buenos nombres
df_clientes_activos = df.filter(col("estado") == "activo")
total_ingresos_mes = df.agg(sum("monto")).collect()[0][0]

# ❌ Malos nombres
df2 = df.filter(col("estado") == "activo")  # ¿Qué es df2?
x = df.agg(sum("monto")).collect()[0][0]   # ¿Qué representa x?
```

#### **Funciones Reutilizables**

```python
def calcular_rfm(df_transacciones: DataFrame, fecha_referencia: str) -> DataFrame:
    """
    Calcula métricas RFM (Recencia, Frecuencia, Monetario) por cliente.
    
    Args:
        df_transacciones: DataFrame con columnas [id_cliente, fecha_compra, monto]
        fecha_referencia: Fecha de referencia en formato 'YYYY-MM-DD'
    
    Returns:
        DataFrame con columnas [id_cliente, recencia, frecuencia, monetario]
    
    Example:
        >>> df_rfm = calcular_rfm(df_ventas, '2026-07-27')
    """
    from pyspark.sql.functions import count, sum, datediff, max, lit, to_date
    
    fecha_ref = to_date(lit(fecha_referencia))
    
    return (
        df_transacciones
        .groupBy("id_cliente")
        .agg(
            datediff(fecha_ref, max("fecha_compra")).alias("recencia"),
            count("id_transaccion").alias("frecuencia"),
            sum("monto").alias("monetario")
        )
    )

# Uso
df_rfm = calcular_rfm(df_ventas, '2026-07-27')
```

### Validaciones y Data Quality

```python
def validar_calidad_datos(df: DataFrame, nombre_tabla: str):
    """
    Ejecuta chequeos de calidad de datos y alerta sobre problemas.
    """
    print(f"\n=== Validación: {nombre_tabla} ===")
    
    # Total de filas
    total = df.count()
    print(f"✅ Total filas: {total:,}")
    
    if total == 0:
        raise ValueError(f"⚠️ CRITICAL: {nombre_tabla} está vacía")
    
    # Duplicados
    duplicados = df.count() - df.dropDuplicates().count()
    if duplicados > 0:
        print(f"⚠️ Warning: {duplicados:,} filas duplicadas ({duplicados/total*100:.1f}%)")
    
    # Nulos por columna
    for col_name in df.columns:
        nulos = df.filter(col(col_name).isNull()).count()
        if nulos > 0:
            pct = nulos / total * 100
            nivel = "⚠️" if pct > 10 else "ℹ️"
            print(f"{nivel} {col_name}: {nulos:,} nulos ({pct:.1f}%)")
    
    print("=" * 40)

# Uso
validar_calidad_datos(df_silver_ventas, "silver.ventas")
```

### Manejo de Errores

```python
try:
    # Operación riesgosa
    df_result = df.join(df_otro, "id", "inner")
    
    # Validación post-join
    if df_result.count() == 0:
        raise ValueError("Join resultó en tabla vacía - revisar claves")
    
    df_result.write.format("delta").mode("overwrite").save("/gold/tabla")
    print("✅ Escritura exitosa")
    
except Exception as e:
    print(f"❌ Error en pipeline: {str(e)}")
    # Loggear error
    import traceback
    traceback.print_exc()
    # Re-lanzar para que el job falle
    raise
```

### Optimización de Performance

#### **1. Particionamiento**

```python
# Particionar por fecha para queries temporales eficientes
df.write.format("delta") \
  .partitionBy("anio", "mes") \
  .mode("overwrite") \
  .save("/gold/ventas_particionadas")

# Query solo lee particiones necesarias
df_q1 = spark.read.format("delta").load("/gold/ventas_particionadas") \
  .filter((col("anio") == 2026) & (col("mes").isin([1, 2, 3])))
```

#### **2. Z-Ordering**

```python
# Optimizar para queries frecuentes por cliente
spark.sql("""
  OPTIMIZE gold.transacciones
  ZORDER BY (id_cliente)
""")

# Queries por cliente serán más rápidas
df_cliente = spark.read.table("gold.transacciones") \
  .filter(col("id_cliente") == 12345)
```

#### **3. Caching Inteligente**

```python
# Cache DataFrame reutilizado múltiples veces
df_base = spark.read.format("delta").load("/silver/ventas")
df_base.cache()  # Almacena en memoria

# Múltiples transformaciones sobre df_base
df_por_mes = df_base.groupBy("mes").agg(sum("monto"))
df_por_categoria = df_base.groupBy("categoria").agg(sum("monto"))
df_por_cliente = df_base.groupBy("id_cliente").agg(sum("monto"))

# Liberar cache cuando ya no se necesita
df_base.unpersist()
```

#### **4. Broadcast Joins**

```python
from pyspark.sql.functions import broadcast

# Para tablas pequeñas (<10MB), broadcast evita shuffle
df_grande = spark.read.table("ventas")  # Millones de filas
df_pequena = spark.read.table("categorias")  # 100 filas

df_joined = df_grande.join(
    broadcast(df_pequena),  # Envía copia a cada worker
    "id_categoria",
    "left"
)
```

### Control de Versiones y CI/CD

**Git workflow para notebooks:**

```bash
# Exportar notebook como .py para Git
databricks workspace export /Users/me/notebook.py notebook.py

# Commit
git add notebook.py
git commit -m "feat: agregar feature engineering RFM"
git push origin main
```

**Databricks Repos** (recomendado):
* Sincronización automática con GitHub/GitLab/Azure DevOps
* Pull requests para code review
* CI/CD integrado

### Monitoreo en Producción
```python
import time

# Instrumentación básica
start_time = time.time()

try:
    # Pipeline
    df_result = ejecutar_pipeline()
    
    # Métricas
    filas_procesadas = df_result.count()
    duracion = time.time() - start_time
    
    # Loggear
    print(f"✅ Pipeline exitoso")
    print(f"   Filas: {filas_procesadas:,}")
    print(f"   Duración: {duracion:.1f}s")
    print(f"   Throughput: {filas_procesadas/duracion:.0f} filas/s")
    
    # Escribir métricas a tabla de monitoreo
    metrics_df = spark.createDataFrame([{
        "timestamp": current_timestamp(),
        "pipeline": "ventas_etl",
        "status": "success",
        "filas": filas_procesadas,
        "duracion_seg": duracion
    }])
    
    metrics_df.write.format("delta").mode("append").saveAsTable("monitoring.pipeline_metrics")
    
except Exception as e:
    duracion = time.time() - start_time
    print(f"❌ Pipeline fallido después de {duracion:.1f}s")
    
    # Loggear fallo
    error_df = spark.createDataFrame([{
        "timestamp": current_timestamp(),
        "pipeline": "ventas_etl",
        "status": "failed",
        "error_message": str(e),
        "duracion_seg": duracion
    }])
    
    error_df.write.format("delta").mode("append").saveAsTable("monitoring.pipeline_metrics")
    raise
```

### Checklist de Producción
✅ **Código modularizado** (funciones, no scripts monolíticos)  
✅ **Validaciones de calidad** en cada capa (Bronze → Silver → Gold)  
✅ **Manejo de errores** con logs descriptivos  
✅ **Optimizaciones** (particionamiento, z-order, broadcast)  
✅ **Documentación** (README, docstrings, comentarios)  
✅ **Tests** (al menos smoke tests)  
✅ **Monitoreo** (métricas de ejecución, alertas)  
✅ **Control de versiones** (Git/Databricks Repos)  
✅ **Automatización** (Jobs programados, no ejecución manual)  
✅ **Rollback plan** (cómo revertir si algo falla)

## 📚 Resumen y Próximos Pasos

### ✅ Conceptos Clave Aprendidos

1. **Proyectos Integradores**: End-to-end desde datos crudos hasta recomendaciones accionables
2. **Arquitectura Medallion**: Bronze (raw) → Silver (cleaned) → Gold (business)
3. **Pipeline ETL**: Extract, Transform, Load con orquestación y monitoreo
4. **Feature Engineering**: Temporales, agregaciones, categóricas, interacciones, texto
5. **Análisis Predictivo**: Clasificación, regresión, evaluación de modelos
6. **Análisis Prescriptivo**: Reglas, optimización, simulación (what-if)
7. **Dashboards Ejecutivos**: KPIs, tendencias, interactividad, storytelling
8. **Documentación**: README, notebooks narrativos, presentaciones ejecutivas
9. **Producción**: Código limpio, validaciones, optimización, monitoreo, CI/CD

---

### 🚀 Preparación para TP07 y TP08

**TP07 - Pipeline Integrador:**
* Implementar arquitectura Bronze-Silver-Gold completa
* ETL automatizado con validaciones
* Feature engineering para ML
* Dashboard con KPIs y tendencias

**TP08 - Proyecto Final:**
* Análisis descriptivo completo (EDA)
* Modelo predictivo (clasificación o regresión)
* Recomendaciones prescriptivas priorizadas
* Dashboard ejecutivo interactivo
* Presentación profesional con storytelling
* Documentación completa (README, notebooks narrativos)

---

### 📝 Plantilla de Proyecto Integrador

**Estructura recomendada:**

```
Proyecto_Nombre/
├── README.md                    # Descripción del proyecto
├── data/
│   ├── bronze/                  # Datos crudos
│   ├── silver/                  # Datos limpios
│   └── gold/                    # Features y agregaciones
├── notebooks/
│   ├── 00_Contexto_Negocio.md   # Problema y objetivos
│   ├── 01_ETL_Bronze.py         # Carga de datos crudos
│   ├── 02_ETL_Silver.py         # Limpieza y validación
│   ├── 03_EDA.py                # Análisis exploratorio
│   ├── 04_Feature_Engineering.py
│   ├── 05_Modelado_ML.py        # Entrenamiento y evaluación
│   ├── 06_Scoring_Prescriptivo.py
│   └── 07_ETL_Gold.py           # Agregaciones finales
├── dashboards/
│   └── Dashboard_Ejecutivo.sql
├── presentacion/
│   └── Presentacion_Final.pdf
└── docs/
    ├── diccionario_datos.md
    └── metodologia.md
```

---

### 📚 Recursos Adicionales

**Documentación:**
* [Delta Lake Best Practices](https://docs.databricks.com/delta/best-practices.html)
* [PySpark MLlib Guide](https://spark.apache.org/docs/latest/ml-guide.html)
* [Databricks SQL Analytics](https://docs.databricks.com/sql/index.html)

**Ejemplos de Proyectos:**
* [Databricks Solution Accelerators](https://www.databricks.com/solutions/accelerators)
* [Kaggle Competitions](https://www.kaggle.com/competitions) (inspiración para datasets)

**Libros recomendados:**
* "Storytelling with Data" - Cole Nussbaumer Knaflic
* "Designing Data-Intensive Applications" - Martin Kleppmann
* "The Data Warehouse Toolkit" - Ralph Kimball

---

### 💬 Preguntas de Reflexión

1. ¿Cómo diseñarías la arquitectura Medallion para un caso de uso de streaming en tiempo real?
2. ¿Qué features ingeniarías para predecir el churn en una empresa de telecomunicaciones?
3. ¿Cómo comunicarías un modelo de ML complejo (e.g., XGBoost) a stakeholders no técnicos?
4. ¿Qué métricas de monitoreo implementarías para un pipeline ETL en producción?

---

### ✨ Mensaje Final

**Un proyecto integrador exitoso:**
* ✅ Resuelve un problema de negocio real
* ✅ Aplica rigor técnico (validaciones, tests, optimización)
* ✅ Comunica insights de forma clara y accionable
* ✅ Está listo para producción (automatizado, monitoreado, documentado)

> “Los mejores proyectos de datos no son los que usan los algoritmos más sofisticados, sino los que generan el mayor impacto de negocio.”

---

**¡Listo para construir tu proyecto integrador en TP07 y TP08! 🚀📊**